<div align="center">
    <img src="img/1.png">
</div>


### 🛠️ Step 1: Environment Initialization
Here, we import our core libraries (`pandas`, `os`). And, we also import `ACSDataSource` and `ACSIncome` from the `folktables` package. This allows us to fetch authentic, unmodified US Census data directly from the source, guaranteeing the integrity of our dataset.

<div align="center">
    <img src="img/2.png">
</div>


In [3]:
import os
import pandas as pd
from folktables import ACSDataSource, ACSIncome

### 📥 Step 2: Data Gathering

In this step, we check if our raw dataset (`ACSIncome_2018_US.csv`) exists locally. If it does not, we leverage `folktables` to download the authentic 2018 American Community Survey data.

<div align="center">
    <img src="img/3.png">
</div>



In [4]:
filename = "ACSIncome_2018_US.csv"
if os.path.exists(filename):
    print(f"Found {filename}")
else:
    script_dir = os.getcwd()

    data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')

    all_states = [
        "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "ID", "IL", 
        "IN", "IA", "KS", "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", 
        "NE", "NV", "NH", "NJ", "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", 
        "SC", "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY", "PR"
    ]

    acs_data = data_source.get_data(states=all_states, download=True)

    X, y, g = ACSIncome.df_to_pandas(acs_data)

    X['PINCP'] = y

    filename = "ACSIncome_2018_US.csv"
    output_path = os.path.join(script_dir, filename)

    X.to_csv(output_path, index=False)

df = pd.read_csv('ACSIncome_2018_US.csv')



data\2018\1-Year\csv_ppa.zip may be corrupted. Please try deleting it and rerunning this command.

Exception:  [WinError 32] The process cannot access the file because it is being used by another process: 'data\\2018\\1-Year\\csv_ppa.zip'


### 🏷️ Step 3: Making Dataset Readable
Raw government data is notoriously cryptic, relying on obscure acronyms (e.g., `AGEP` instead of age, `COW` instead of workclass). 

To ensure our downstream fairness auditing algorithms are interpretable, we translate these headers into clear, human-readable names.

<div align="center">
    <img src="img/4.jpg">
</div>



In [5]:

col_dict = {
    'AGEP': 'age',
    'COW': 'workclass',
    'SCHL': 'education',
    'MAR': 'marital-status',
    'OCCP': 'occupation',
    'POBP': 'place-of-birth',
    'RELP': 'relationship',
    'WKHP': 'hours-per-week',
    'SEX': 'sex',
    'RAC1P': 'race',
    'PINCP': 'income'
    }
df.rename(columns=col_dict, inplace=True)
print(df.head())

    age  workclass  education  marital-status  occupation  place-of-birth  \
0  18.0        1.0       18.0             5.0      4720.0            13.0   
1  53.0        5.0       17.0             5.0      3605.0            18.0   
2  41.0        1.0       16.0             5.0      7330.0             1.0   
3  18.0        6.0       18.0             5.0      2722.0             1.0   
4  21.0        5.0       19.0             5.0      3870.0            12.0   

   relationship  hours-per-week  sex  race  income  
0          17.0            21.0  2.0   2.0   False  
1          16.0            40.0  1.0   1.0   False  
2          17.0            40.0  1.0   1.0   False  
3          17.0             2.0  2.0   1.0   False  
4          17.0            50.0  1.0   1.0   False  


### 🗺️ Step 4: Demographic Decoding
Just like the column headers, the actual data values are encoded as raw numbers (e.g., "1" representing "Private For-Profit"). 

Here, we define exhaustive dictionaries to decode these values. This ensures that when we later perform **Controlled Bias Injection** (e.g., intentionally skewing data against certain groups), we are targeting the correct demographics information.

<div align="center">
    <img src="img/5.png">
</div>


In [6]:

cow_dict = {
    "1": "Private For-Profit",
    "2": "Private Non-Profit",
    "3": "Local Government",
    "4": "State Government",
    "5": "Federal Government",
    "6": "Self-Employed (Not Inc.)",
    "7": "Self-Employed (Inc.)",
    "8": "Without Pay",
    "9": "Unemployed"
}
df['workclass'] = df['workclass'].map(lambda x: cow_dict.get(str(int(x))) if pd.notna(x) else "N/A")

schl_dict = {
    "1": "No schooling completed",
    "2": "Nursery school, preschool",
    "3": "Kindergarten",
    "4": "Grade 1",
    "5": "Grade 2",
    "6": "Grade 3",
    "7": "Grade 4",
    "8": "Grade 5",
    "9": "Grade 6",
    "10": "Grade 7",
    "11": "Grade 8",
    "12": "Grade 9",
    "13": "Grade 10",
    "14": "Grade 11",
    "15": "12th grade, no diploma",
    "16": "Regular high school diploma",
    "17": "GED or alternative credential",
    "18": "Some college, but less than 1 year",
    "19": "One or more years of college credit, no degree",
    "20": "Associate's degree",
    "21": "Bachelor's degree",
    "22": "Master's degree",
    "23": "Professional degree beyond a bachelor's degree",
    "24": "Doctorate degree"
}
df['education'] = df['education'].map(lambda x: schl_dict.get(str(int(x))) if pd.notna(x) else "N/A")

mar_dict = {
    "1": "Married",
    "2": "Widowed",
    "3": "Divorced",
    "4": "Separated",
    "5": "Never married"
}
df['marital-status'] = df['marital-status'].map(lambda x: mar_dict.get(str(int(x))) if pd.notna(x) else "N/A")

occp_dict = {
    "10": "Chief Executives And Legislators",
    "20": "General And Operations Managers",
    "40": "Advertising And Promotions Managers",
    "51": "Marketing Managers",
    "52": "Sales Managers",
    "60": "Public Relations And Fundraising Managers",
    "101": "Administrative Services Managers",
    "102": "Facilities Managers",
    "110": "Computer And Information Systems Managers",
    "120": "Financial Managers",
    "135": "Compensation And Benefits Managers",
    "136": "Human Resources Managers",
    "137": "Training And Development Managers",
    "140": "Industrial Production Managers",
    "150": "Purchasing Managers",
    "160": "Transportation, Storage, And Distribution Managers",
    "205": "Farmers, Ranchers, And Agricultural Managers",
    "220": "Construction Managers",
    "230": "Education And Childcare Administrators",
    "300": "Architectural And Engineering Managers",
    "310": "Food Service Managers",
    "335": "Entertainment and Recreation Managers",
    "340": "Lodging Managers",
    "350": "Medical And Health Services Managers",
    "360": "Natural Sciences Managers",
    "410": "Property, Real Estate, And Community Association Managers",
    "420": "Social And Community Service Managers",
    "425": "Emergency Management Directors",
    "440": "Managers",
    "500": "Agents And Business Managers Of Artists, Performers, And Athletes",
    "510": "Buyers And Purchasing Agents, Farm Products",
    "520": "Wholesale And Retail Buyers, Except Farm Products",
    "530": "Purchasing Agents, Except Wholesale, Retail, And Farm Products",
    "540": "Claims Adjusters, Appraisers, Examiners, And Investigators",
    "565": "Compliance Officers",
    "600": "Cost Estimators",
    "630": "Human Resources Workers",
    "640": "Compensation, Benefits, And Job Analysis Specialists",
    "650": "Training And Development Specialists",
    "700": "Logisticians",
    "705": "Project Management Specialists",
    "710": "Management Analysts",
    "725": "Meeting, Convention, And Event Planners",
    "726": "Fundraisers",
    "735": "Market Research Analysts And Marketing Specialists",
    "750": "Business Operations Specialists, All Other",
    "800": "Accountants And Auditors",
    "810": "Property Appraisers and Assessors",
    "820": "Budget Analysts",
    "830": "Credit Analysts",
    "845": "Financial And Investment Analysts",
    "850": "Personal Financial Advisors",
    "860": "Insurance Underwriters",
    "900": "Financial Examiners",
    "910": "Credit Counselors And Loan Officers",
    "930": "Tax Examiners And Collectors, And Revenue Agents",
    "940": "Tax Preparers",
    "960": "Financial Specialists",
    "1005": "Computer And Information Research Scientists",
    "1006": "Computer Systems Analysts",
    "1007": "Information Security Analysts",
    "1010": "Computer Programmers",
    "1021": "Software Developers",
    "1022": "Software Quality Assurance Analysts and Testers",
    "1031": "Web Developers",
    "1032": "Web And Digital Interface Designers",
    "1050": "Computer Support Specialists",
    "1065": "Database Administrators and Architects",
    "1105": "Network And Computer Systems Administrators",
    "1106": "Computer Network Architects",
    "1108": "Computer Occupations, All Other",
    "1200": "Actuaries",
    "1220": "Operations Research Analysts",
    "1240": "Mathematical Science Occupations",
    "1305": "Architects, Except Landscape And Naval",
    "1306": "Landscape Architects",
    "1310": "Surveyors, Cartographers, And Photogrammetrists",
    "1320": "Aerospace Engineers",
    "1340": "Biomedical And Agricultural Engineers",
    "1350": "Chemical Engineers",
    "1360": "Civil Engineers",
    "1400": "Computer Hardware Engineers",
    "1410": "Electrical And Electronics Engineers",
    "1420": "Environmental Engineers",
    "1430": "Industrial Engineers, Including Health And Safety",
    "1440": "Marine Engineers And Naval Architects",
    "1450": "Materials Engineers",
    "1460": "Mechanical Engineers",
    "1520": "Petroleum, Mining And Geological Engineers, Including Mining Safety Engineers",
    "1530": "Engineers",
    "1541": "Architectural And Civil Drafters",
    "1545": "Drafters",
    "1551": "Electrical And Electronic Engineering Technologists and Technicians",
    "1555": "Engineering Technologists And Technicians, Except Drafters",
    "1560": "Surveying And Mapping Technicians",
    "1600": "Agricultural And Food Scientists",
    "1610": "Biological Scientists",
    "1640": "Conservation Scientists And Foresters",
    "1650": "Life Scientists",
    "1700": "Astronomers And Physicists",
    "1710": "Atmospheric And Space Scientists",
    "1720": "Chemists And Materials Scientists",
    "1745": "Environmental Scientists And Specialists, Including Health",
    "1750": "Geoscientists And Hydrologists, Except Geographers",
    "1760": "Physical Scientists, All Other",
    "1800": "Economists",
    "1821": "Clinical And Counseling Psychologists",
    "1822": "School Psychologists",
    "1825": "Psychologists",
    "1840": "Urban And Regional Planners",
    "1860": "Social Scientists",
    "1900": "Agricultural And Food Science Technicians",
    "1910": "Biological Technicians",
    "1920": "Chemical Technicians",
    "1935": "Environmental Science and Geoscience Technicians, And Nuclear Technicians",
    "1970": "Life, Physical, And Social Science Technicians",
    "1980": "Occupational Health And Safety Specialists and Technicians",
    "2001": "Substance Abuse And Behavioral Disorder Counselors",
    "2002": "Educational, Guidance, And Career Counselors And Advisors",
    "2003": "Marriage And Family Therapists",
    "2004": "Mental Health Counselors",
    "2005": "Rehabilitation Counselors",
    "2006": "Counselors, All Other",
    "2011": "Child, Family, And School Social Workers",
    "2012": "Healthcare Social Workers",
    "2013": "Mental Health And Substance Abuse Social Workers",
    "2014": "Social Workers, All Other",
    "2015": "Probation Officers And Correctional Treatment Specialists",
    "2016": "Social And Human Service Assistants",
    "2025": "Community and Social Service Specialists",
    "2040": "Clergy",
    "2050": "Directors, Religious Activities And Education",
    "2060": "Religious Workers, All Other",
    "2100": "Lawyers, And Judges, Magistrates, And Judicial Workers",
    "2105": "Judicial Law Clerks",
    "2145": "Paralegals And Legal Assistants",
    "2170": "Title Examiners, Abstractors, and Searchers",
    "2180": "Legal Support Workers, All Other",
    "2205": "Postsecondary Teachers",
    "2300": "Preschool And Kindergarten Teachers",
    "2310": "Elementary And Middle School Teachers",
    "2320": "Secondary School Teachers",
    "2330": "Special Education Teachers",
    "2350": "Tutors",
    "2360": "Teachers and Instructors",
    "2400": "Archivists, Curators, And Museum Technicians",
    "2435": "Librarians And Media Collections Specialists",
    "2440": "Library Technicians",
    "2545": "Teaching Assistants",
    "2555": "Educational Instruction and Library Workers",
    "2600": "Artists And Related Workers",
    "2631": "Commercial And Industrial Designers",
    "2632": "Fashion Designers",
    "2633": "Floral Designers",
    "2634": "Graphic Designers",
    "2635": "Interior Designers",
    "2636": "Merchandise Displayers And Windows Trimmers",
    "2640": "Designers",
    "2700": "Actors",
    "2710": "Producers And Directors",
    "2721": "Athletes and Sports Competitors",
    "2722": "Coaches and Scouts",
    "2723": "Umpires, Referees, And Sports Officials",
    "2740": "Dancers And Choreographers",
    "2751": "Music Directors and Composers",
    "2752": "Musicians and Singers",
    "2755": "Disc Jockeys, Except Radio",
    "2770": "Entertainers And Performers, Sports and Related Workers, All Other",
    "2805": "Broadcast Announcers And Radio Disc Jockeys",
    "2810": "News Analysts, Reporters And Correspondents",
    "2825": "Public Relations Specialists",
    "2830": "Editors",
    "2840": "Technical Writers",
    "2850": "Writers And Authors",
    "2861": "Interpreters and Translators",
    "2862": "Court Reporters and Simultaneous Captioners",
    "2865": "Media And Communication Workers, All Other",
    "2905": "Media And Communication Equipment Workers",
    "2910": "Photographers",
    "2920": "Television, Video, And Motion Picture Camera Operators And Editors",
    "3000": "Chiropractors",
    "3010": "Dentists",
    "3030": "Dietitians And Nutritionists",
    "3040": "Optometrists",
    "3050": "Pharmacists",
    "3090": "Physicians",
    "3100": "Surgeons",
    "3110": "Physician Assistants",
    "3120": "Podiatrists",
    "3140": "Audiologists",
    "3150": "Occupational Therapists",
    "3160": "Physical Therapists",
    "3200": "Radiation Therapists",
    "3210": "Recreational Therapists",
    "3220": "Respiratory Therapists",
    "3230": "Speech-Language Pathologists",
    "3245": "Therapists",
    "3250": "Veterinarians",
    "3255": "Registered Nurses",
    "3256": "Nurse Anesthetists",
    "3258": "Nurse Practitioners, And Nurse Midwives",
    "3261": "Acupuncturists",
    "3270": "Healthcare Diagnosing Or Treating Practitioners, All Other",
    "3300": "Clinical Laboratory Technologists And Technicians",
    "3310": "Dental Hygienists",
    "3321": "Cardiovascular Technologists and Technicians",
    "3322": "Diagnostic Medical Sonographers",
    "3323": "Radiologic Technologists And Technicians",
    "3324": "Magnetic Resonance Imaging Technologists",
    "3330": "Nuclear Medicine Technologists and Medical Dosimetrists",
    "3401": "Emergency Medical Technicians",
    "3402": "Paramedics",
    "3421": "Pharmacy Technicians",
    "3422": "Psychiatric Technicians",
    "3423": "Surgical Technologists",
    "3424": "Veterinary Technologists and Technicians",
    "3430": "Dietetic Technicians And Ophthalmic Medical Technicians",
    "3500": "Licensed Practical And Licensed Vocational Nurses",
    "3515": "Medical Records Specialists",
    "3520": "Opticians, Dispensing",
    "3545": "Miscellaneous Health Technologists and Technicians",
    "3550": "Healthcare Practitioners and Technical Occupations",
    "3601": "Home Health Aides",
    "3602": "Personal Care Aides",
    "3603": "Nursing Assistants",
    "3605": "Orderlies and Psychiatric Aides",
    "3610": "Occupational Therapy Assistants And Aides",
    "3620": "Physical Therapist Assistants And Aides",
    "3630": "Massage Therapists",
    "3640": "Dental Assistants",
    "3645": "Medical Assistants",
    "3646": "Medical Transcriptionists",
    "3647": "Pharmacy Aides",
    "3648": "Veterinary Assistants And Laboratory Animal Caretakers",
    "3649": "Phlebotomists",
    "3655": "Healthcare Support Workers",
    "3700": "First-Line Supervisors Of Correctional Officers",
    "3710": "First-Line Supervisors Of Police And Detectives",
    "3720": "First-Line Supervisors Of Fire Fighting And Prevention Workers",
    "3725": "First-Line Supervisors of Security And Protective Service Workers, All Other",
    "3740": "Firefighters",
    "3750": "Fire Inspectors",
    "3801": "Bailiffs",
    "3802": "Correctional Officers and Jailers",
    "3820": "Detectives And Criminal Investigators",
    "3840": "Fish And Game Wardens And Parking Enforcement Officers",
    "3870": "Police Officers",
    "3900": "Animal Control Workers",
    "3910": "Private Detectives And Investigators",
    "3930": "Security Guards And Gaming Surveillance Officers",
    "3940": "Crossing Guards And Flaggers",
    "3945": "Transportation Security Screeners",
    "3946": "School Bus Monitors",
    "3960": "Protective Service Workers",
    "4000": "Chefs And Head Cooks",
    "4010": "First-Line Supervisors Of Food Preparation And Serving Workers",
    "4020": "Cooks",
    "4030": "Food Preparation Workers",
    "4040": "Bartenders",
    "4055": "Fast Food And Counter Workers",
    "4110": "Waiters And Waitresses",
    "4120": "Food Servers, Nonrestaurant",
    "4130": "Dining Room And Cafeteria Attendants And Bartender Helpers",
    "4140": "Dishwashers",
    "4150": "Hosts And Hostesses, Restaurant, Lounge, And Coffee Shop",
    "4160": "Food Preparation and Serving Related Workers, All Other",
    "4200": "First-Line Supervisors Of Housekeeping And Janitorial Workers",
    "4210": "First-Line Supervisors Of Landscaping, Lawn Service, And Groundskeeping Workers",
    "4220": "Janitors And Building Cleaners",
    "4230": "Maids And Housekeeping Cleaners",
    "4240": "Pest Control Workers",
    "4251": "Landscaping And Groundskeeping Workers",
    "4252": "Tree Trimmers and Pruners",
    "4255": "Grounds Maintenance Workers",
    "4330": "Supervisors Of Personal Care And Service Workers",
    "4340": "Animal Trainers",
    "4350": "Animal Caretakers",
    "4400": "Gambling Services Workers",
    "4420": "Ushers, Lobby Attendants, And Ticket Takers",
    "4435": "Entertainment Attendants And Related Workers",
    "4461": "Embalmers, Crematory Operators, And Funeral Attendants",
    "4465": "Morticians, Undertakers, And Funeral Arrangers",
    "4500": "Barbers",
    "4510": "Hairdressers, Hairstylists, And Cosmetologists",
    "4521": "Manicurists And Pedicurists",
    "4522": "Skincare Specialists",
    "4525": "Personal Appearance Workers",
    "4530": "Baggage Porters, Bellhops, And Concierges",
    "4540": "Tour And Travel Guides",
    "4600": "Childcare Workers",
    "4621": "Exercise Trainers And Group Fitness Instructors",
    "4622": "Recreation Workers",
    "4640": "Residential Advisors",
    "4655": "Personal Care and Service Workers, All Other",
    "4700": "First-Line Supervisors Of Retail Sales Workers",
    "4710": "First-Line Supervisors Of Non-Retail Sales Workers",
    "4720": "Cashiers",
    "4740": "Counter And Rental Clerks",
    "4750": "Parts Salespersons",
    "4760": "Retail Salespersons",
    "4800": "Advertising Sales Agents",
    "4810": "Insurance Sales Agents",
    "4820": "Securities, Commodities, And Financial Services Sales Agents",
    "4830": "Travel Agents",
    "4840": "Sales Representatives Of Services, Except Advertising, Insurance, Financial Services, And Travel",
    "4850": "Sales Representatives, Wholesale And Manufacturing",
    "4900": "Models, Demonstrators, And Product Promoters",
    "4920": "Real Estate Brokers And Sales Agents",
    "4930": "Sales Engineers",
    "4940": "Telemarketers",
    "4950": "Door-To-Door Sales Workers, News And Street Vendors, And Related Workers",
    "4965": "Sales And Related Workers, All Other",
    "5000": "First-Line Supervisors Of Office And Administrative Support Workers",
    "5010": "Switchboard Operators, Including Answering Service",
    "5020": "Telephone Operators",
    "5040": "Communications Equipment Operators, All Other",
    "5100": "Bill And Account Collectors",
    "5110": "Billing And Posting Clerks",
    "5120": "Bookkeeping, Accounting, And Auditing Clerks",
    "5140": "Payroll And Timekeeping Clerks",
    "5150": "Procurement Clerks",
    "5160": "Tellers",
    "5165": "Financial Clerks",
    "5220": "Court, Municipal, And License Clerks",
    "5230": "Credit Authorizers, Checkers, And Clerks",
    "5240": "Customer Service Representatives",
    "5250": "Eligibility Interviewers, Government Programs",
    "5260": "File Clerks",
    "5300": "Hotel, Motel, And Resort Desk Clerks",
    "5310": "Interviewers, Except Eligibility And Loan",
    "5320": "Library Assistants, Clerical",
    "5330": "Loan Interviewers And Clerks",
    "5340": "New Accounts Clerks",
    "5350": "Correspondence Clerks And Order Clerks",
    "5360": "Human Resources Assistants, Except Payroll And Timekeeping",
    "5400": "Receptionists And Information Clerks",
    "5410": "Reservation And Transportation Ticket Agents And Travel Clerks",
    "5420": "Information And Records Clerks",
    "5500": "Cargo And Freight Agents",
    "5510": "Couriers And Messengers",
    "5521": "Public Safety Telecommunicators",
    "5522": "Dispatchers, Except Police, Fire, And Ambulance",
    "5530": "Meter Readers, Utilities",
    "5540": "Postal Service Clerks",
    "5550": "Postal Service Mail Carriers",
    "5560": "Postal Service Mail Sorters, Processors, And Processing Machine Operators",
    "5600": "Production, Planning, And Expediting Clerks",
    "5610": "Shipping, Receiving, And Inventory Clerks",
    "5630": "Weighers, Measurers, Checkers, And Samplers, Recordkeeping",
    "5710": "Executive Secretaries And Executive Administrative Assistants",
    "5720": "Legal Secretaries and Administrative Assistants",
    "5730": "Medical Secretaries and Administrative Assistants",
    "5740": "Secretaries And Administrative Assistants, Except Legal, Medial, And Executive",
    "5810": "Data Entry Keyers",
    "5820": "Word Processors And Typists",
    "5840": "Insurance Claims And Policy Processing Clerks",
    "5850": "Mail Clerks And Mail Machine Operators, Except Postal Service",
    "5860": "Office Clerks, General",
    "5900": "Office Machine Operators, Except Computer",
    "5910": "Proofreaders And Copy Markers",
    "5920": "Statistical Assistants",
    "5940": "Office And Administrative Support Workers",
    "6005": "First-Line Supervisors Of Farming, Fishing, And Forestry Workers",
    "6010": "Agricultural Inspectors",
    "6040": "Graders And Sorters, Agricultural Products",
    "6050": "Agricultural Workers",
    "6115": "Fishing And Hunting Workers",
    "6120": "Forest And Conservation Workers",
    "6130": "Logging Workers",
    "6200": "First-Line Supervisors Of Construction Trades And Extraction Workers",
    "6210": "Boilermakers",
    "6220": "Brickmasons, Blockmasons, Stonemasons, And Reinforcing Iron And Rebar Workers",
    "6230": "Carpenters",
    "6240": "Carpet, Floor, And Tile Installers And Finishers",
    "6250": "Cement Masons, Concrete Finishers, And Terrazzo Workers",
    "6260": "Construction Laborers",
    "6305": "Construction Equipment Operators",
    "6330": "Drywall Installers, Ceiling Tile Installers, And Tapers",
    "6355": "Electricians",
    "6360": "Glaziers",
    "6400": "Insulation Workers",
    "6410": "Painters and Paperhangers",
    "6441": "Pipelayers",
    "6442": "Plumbers, Pipefitters, And Steamfitters",
    "6460": "Plasterers And Stucco Masons",
    "6515": "Roofers",
    "6520": "Sheet Metal Workers",
    "6530": "Structural Iron And Steel Workers",
    "6540": "Solar Photovoltaic Installers",
    "6600": "Helpers, Construction Trades",
    "6660": "Construction And Building Inspectors",
    "6700": "Elevator Installers And Repairers",
    "6710": "Fence Erectors",
    "6720": "Hazardous Materials Removal Workers",
    "6730": "Highway Maintenance Workers",
    "6740": "Rail-Track Laying And Maintenance Equipment Operators",
    "6765": "Construction And Related Workers",
    "6800": "Derrick, Rotary Drill, And Service Unit Operators, And Roustabouts, Oil, Gas, And Mining",
    "6825": "Surface Mining Machine Operators And Earth Drillers",
    "6835": "Explosives Workers, Ordnance Handling Experts, and Blasters",
    "6850": "Underground Mining Machine Operators",
    "6950": "Extraction Workers",
    "7000": "First-Line Supervisors Of Mechanics, Installers, And Repairers",
    "7010": "Computer, Automated Teller, And Office Machine Repairers",
    "7020": "Radio And Telecommunications Equipment Installers And Repairers",
    "7030": "Avionics Technicians",
    "7040": "Electric Motor, Power Tool, And Related Repairers",
    "7100": "Electrical And Electronic Equipment Mechanics, Installers, And Repairers.",
    "7120": "Electronic Home Entertainment Equipment Installers And Repairers",
    "7130": "Security And Fire Alarm Systems Installers",
    "7140": "Aircraft Mechanics And Service Technicians",
    "7150": "Automotive Body And Related Repairers",
    "7160": "Automotive Glass Installers And Repairers",
    "7200": "Automotive Service Technicians And Mechanics",
    "7210": "Bus And Truck Mechanics And Diesel Engine Specialists",
    "7220": "Heavy Vehicle And Mobile Equipment Service Technicians And Mechanics",
    "7240": "Small Engine Mechanics",
    "7260": "Miscellaneous Vehicle And Mobile Equipment Mechanics, Installers, And Repairers",
    "7300": "Control And Valve Installers And Repairers",
    "7315": "Heating, Air Conditioning, And Refrigeration Mechanics And Installers",
    "7320": "Home Appliance Repairers",
    "7330": "Industrial And Refractory Machinery Mechanics",
    "7340": "Maintenance And Repair Workers, General",
    "7350": "Maintenance Workers, Machinery",
    "7360": "Millwrights",
    "7410": "Electrical Power-Line Installers And Repairers",
    "7420": "Telecommunications Line Installers And Repairers",
    "7430": "Precision Instrument And Equipment Repairers",
    "7510": "Coin, Vending, And Amusement Machine Servicers And Repairers",
    "7540": "Locksmiths And Safe Repairers",
    "7560": "Riggers",
    "7610": "Installation, Maintenance, And Repair Workers",
    "7640": "Installation, Maintenance, And Repair Workers",
    "7700": "First-Line Supervisors Of Production And Operating Workers",
    "7720": "Electrical, Electronics, And Electromechanical Assemblers",
    "7730": "Engine And Machine Assemblers",
    "7740": "Structural Metal Fabricators And Fitters",
    "7750": "Assemblers And Fabricators",
    "7800": "Bakers",
    "7810": "Butchers And Meat, Poultry, And Fish Processing Workers",
    "7830": "Food And Tobacco Roasting, Baking, And Drying Machine Operators And Tenders",
    "7840": "Food Batchmakers",
    "7850": "Food Cooking Machine Operators And Tenders",
    "7855": "Food Processing Workers, All Other",
    "7905": "Computer Numerically Controlled Tool Operators And Programmers",
    "7925": "Forming Machine Setters, Operators, And Tenders, Metal And Plastic",
    "7950": "Cutting, Punching, And Press Machine Setters, Operators, And Tenders, Metal And Plastic",
    "8000": "Grinding, Lapping, Polishing, And Buffing Machine Tool",
    "8025": "Machine Tool Setters, Operators, And Tenders, Metal and Plastic",
    "8030": "Machinists",
    "8040": "Metal Furnace Operators, Tenders, Pourers, And Casters",
    "8100": "Model Makers, Patternmakers, And Molding Machine Setters, Metal And Plastic",
    "8130": "Tool And Die Makers",
    "8140": "Welding, Soldering, And Brazing Workers",
    "8225": "Metal Workers And Plastic Workers",
    "8250": "Prepress Technicians And Workers",
    "8255": "Printing Press Operators",
    "8256": "Print Binding And Finishing Workers",
    "8300": "Laundry And Dry-Cleaning Workers",
    "8310": "Pressers, Textile, Garment, And Related Materials",
    "8320": "Sewing Machine Operators",
    "8335": "Shoe And Leather Workers",
    "8350": "Tailors, Dressmakers, And Sewers",
    "8365": "Textile Machine Setters, Operators, And Tenders",
    "8450": "Upholsterers",
    "8465": "Textile, Apparel, And Furnishings Workers",
    "8500": "Cabinetmakers And Bench Carpenters",
    "8510": "Furniture Finishers",
    "8530": "Sawing Machine Setters, Operators, And Tenders, Wood",
    "8540": "Woodworking Machine Setters, Operators, And Tenders, Except Sawing",
    "8555": "Woodworkers",
    "8600": "Power Plant Operators, Distributors, And Dispatchers",
    "8610": "Stationary Engineers And Boiler Operators",
    "8620": "Water And Wastewater Treatment Plant And System Operators",
    "8630": "Miscellaneous Plant And System Operators",
    "8640": "Chemical Processing Machine Setters, Operators, And Tenders",
    "8650": "Crushing, Grinding, Polishing, Mixing, And Blending Workers",
    "8710": "Cutting Workers",
    "8720": "Extruding, Forming, Pressing, And Compacting Machine Setters, Operators, And Tenders",
    "8730": "Furnace, Kiln, Oven, Drier, And Kettle Operators And Tenders",
    "8740": "Inspectors, Testers, Sorters, Samplers, And Weighers",
    "8750": "Jewelers And Precious Stone And Metal Workers",
    "8760": "Dental And Ophthalmic Laboratory Technicians And Medical Appliance Technicians",
    "8800": "Packaging And Filling Machine Operators And Tenders",
    "8810": "Painting Workers",
    "8830": "Photographic Process Workers And Processing Machine Operators",
    "8850": "Adhesive Bonding Machine Operators And Tenders",
    "8910": "Etchers And Engravers",
    "8920": "Molders, Shapers, And Casters, Except Metal And Plastic",
    "8930": "Paper Goods Machine Setters, Operators, And Tenders",
    "8940": "Tire Builders",
    "8950": "Helpers-Production Workers",
    "8990": "Miscellaneous Production Workers, Including Equipment Operators And Tenders",
    "9005": "Supervisors Of Transportation And Material Moving Workers",
    "9030": "Aircraft Pilots And Flight Engineers",
    "9040": "Air Traffic Controllers And Airfield Operations Specialists",
    "9050": "Flight Attendants",
    "9110": "Ambulance Drivers And Attendants, Except Emergency Medical Technicians",
    "9121": "Bus Drivers, School",
    "9122": "Bus Drivers, Transit And Intercity",
    "9130": "Driver/Sales Workers And Truck Drivers",
    "9141": "Shuttle Drivers And Chauffeurs",
    "9142": "Taxi Drivers",
    "9150": "Motor Vehicle Operators, All Other",
    "9210": "Locomotive Engineers And Operators",
    "9240": "Railroad Conductors And Yardmasters",
    "9265": "Rail Transportation Workers",
    "9300": "Sailors And Marine Oilers, And Ship Engineers",
    "9310": "Ship And Boat Captains And Operators",
    "9350": "Parking Lot Attendants",
    "9365": "Transportation Service Attendants",
    "9410": "Transportation Inspectors",
    "9415": "Passenger Attendants",
    "9430": "Transportation Workers",
    "9510": "Crane And Tower Operators",
    "9570": "Conveyor, Dredge, And Hoist and Winch Operators",
    "9600": "Industrial Truck And Tractor Operators",
    "9610": "Cleaners Of Vehicles And Equipment",
    "9620": "Laborers And Freight, Stock, And Material Movers, Hand",
    "9630": "Machine Feeders And Offbearers",
    "9640": "Packers And Packagers, Hand",
    "9645": "Stockers And Order Fillers",
    "9650": "Pumping Station Operators",
    "9720": "Refuse And Recyclable Material Collectors",
    "9760": "Material Moving Workers",
    "9800": "Military Officer Special And Tactical Operations Leaders",
    "9810": "First-Line Enlisted Military Supervisors",
    "9825": "Military Enlisted Tactical Operations And Air/Weapons Specialists And Crew Members",
    "9830": "Military, Rank Not Specified"
}
df['occupation'] = df['occupation'].map(lambda x: occp_dict.get(str(int(x))) if pd.notna(x) else "N/A")

pobp_dict = {
    "1": "Alabama",
    "2": "Alaska",
    "4": "Arizona",
    "5": "Arkansas",
    "6": "California",
    "8": "Colorado",
    "9": "Connecticut",
    "10": "Delaware",
    "11": "District of Columbia",
    "12": "Florida",
    "13": "Georgia",
    "15": "Hawaii",
    "16": "Idaho",
    "17": "Illinois",
    "18": "Indiana",
    "19": "Iowa",
    "20": "Kansas",
    "21": "Kentucky",
    "22": "Louisiana",
    "23": "Maine",
    "24": "Maryland",
    "25": "Massachusetts",
    "26": "Michigan",
    "27": "Minnesota",
    "28": "Mississippi",
    "29": "Missouri",
    "30": "Montana",
    "31": "Nebraska",
    "32": "Nevada",
    "33": "New Hampshire",
    "34": "New Jersey",
    "35": "New Mexico",
    "36": "New York",
    "37": "North Carolina",
    "38": "North Dakota",
    "39": "Ohio",
    "40": "Oklahoma",
    "41": "Oregon",
    "42": "Pennsylvania",
    "44": "Rhode Island",
    "45": "South Carolina",
    "46": "South Dakota",
    "47": "Tennessee",
    "48": "Texas",
    "49": "Utah",
    "50": "Vermont/VT",
    "51": "Virginia/VA",
    "53": "Washington",
    "54": "West Virginia",
    "55": "Wisconsin",
    "56": "Wyoming",
    "60": "American Samoa",
    "66": "Guam",
    "69": "Commonwealth of the Northern Mariana Islands",
    "72": "Puerto Rico",
    "78": "US Virgin Islands",
    "100": "Albania",
    "102": "Austria",
    "103": "Belgium",
    "104": "Bulgaria",
    "105": "Czechoslovakia",
    "106": "Denmark",
    "108": "Finland",
    "109": "France",
    "110": "Germany",
    "116": "Greece",
    "117": "Hungary",
    "118": "Iceland",
    "119": "Ireland",
    "120": "Italy",
    "126": "Netherlands",
    "127": "Norway",
    "128": "Poland",
    "129": "Portugal",
    "130": "Azores Islands",
    "132": "Romania",
    "134": "Spain",
    "136": "Sweden",
    "137": "Switzerland",
    "138": "United Kingdom, Not Specified",
    "139": "England",
    "140": "Scotland",
    "142": "Northern Ireland",
    "147": "Yugoslavia",
    "148": "Czech Republic",
    "149": "Slovakia",
    "150": "Bosnia and Herzegovina",
    "151": "Croatia",
    "152": "Macedonia",
    "154": "Serbia",
    "156": "Latvia",
    "157": "Lithuania",
    "158": "Armenia",
    "159": "Azerbaijan",
    "160": "Belarus",
    "161": "Georgia",
    "162": "Moldova",
    "163": "Russia",
    "164": "Ukraine",
    "165": "USSR",
    "166": "Europe",
    "167": "Kosovo",
    "168": "Montenegro",
    "169": "Other Europe, Not Specified",
    "200": "Afghanistan",
    "202": "Bangladesh",
    "203": "Bhutan",
    "205": "Myanmar",
    "206": "Cambodia",
    "207": "China",
    "209": "Hong Kong",
    "210": "India",
    "211": "Indonesia",
    "212": "Iran",
    "213": "Iraq",
    "214": "Israel",
    "215": "Japan",
    "216": "Jordan",
    "217": "Korea",
    "218": "Kazakhstan",
    "219": "Kyrgyzstan",
    "222": "Kuwait",
    "223": "Laos",
    "224": "Lebanon",
    "226": "Malaysia",
    "228": "Mongolia",
    "229": "Nepal",
    "231": "Pakistan",
    "233": "Philippines",
    "235": "Saudi Arabia",
    "236": "Singapore",
    "238": "Sri Lanka",
    "239": "Syria",
    "240": "Taiwan",
    "242": "Thailand",
    "243": "Turkey",
    "245": "United Arab Emirates",
    "246": "Uzbekistan",
    "247": "Vietnam",
    "248": "Yemen",
    "249": "Asia",
    "253": "South Central Asia, Not Specified",
    "254": "Other Asia, Not Specified",
    "300": "Bermuda",
    "301": "Canada",
    "303": "Mexico",
    "310": "Belize",
    "311": "Costa Rica",
    "312": "El Salvador",
    "313": "Guatemala",
    "314": "Honduras",
    "315": "Nicaragua",
    "316": "Panama",
    "321": "Antigua & Barbuda",
    "323": "Bahamas",
    "324": "Barbados",
    "327": "Cuba",
    "328": "Dominica",
    "329": "Dominican Republic",
    "330": "Grenada",
    "332": "Haiti",
    "333": "Jamaica",
    "338": "St. Kitts-Nevis",
    "339": "St. Lucia",
    "340": "St. Vincent & the Grenadines",
    "341": "Trinidad & Tobago",
    "343": "West Indies",
    "344": "Caribbean, Not Specified",
    "360": "Argentina",
    "361": "Bolivia",
    "362": "Brazil",
    "363": "Chile",
    "364": "Colombia",
    "365": "Ecuador",
    "368": "Guyana",
    "369": "Paraguay",
    "370": "Peru",
    "372": "Uruguay",
    "373": "Venezuela",
    "374": "South America",
    "399": "Americas, Not Specified",
    "400": "Algeria",
    "407": "Cameroon",
    "408": "Cabo Verde",
    "412": "Congo",
    "414": "Egypt",
    "416": "Ethiopia",
    "417": "Eritrea",
    "420": "Gambia",
    "421": "Ghana",
    "423": "Guinea",
    "425": "Ivory Coast",
    "427": "Kenya",
    "429": "Liberia",
    "430": "Libya",
    "436": "Morocco",
    "440": "Nigeria",
    "442": "Rwanda",
    "444": "Senegal",
    "447": "Sierra Leone",
    "448": "Somalia",
    "449": "South Africa",
    "451": "Sudan",
    "453": "Tanzania",
    "454": "Togo",
    "456": "Tunisia",
    "457": "Uganda",
    "459": "Democratic Republic of Congo (Zaire)",
    "460": "Zambia",
    "461": "Zimbabwe",
    "462": "Africa",
    "463": "South Sudan",
    "464": "Northern Africa, Not Specified",
    "467": "Western Africa, Not Specified",
    "468": "Other Africa, Not Specified",
    "469": "Eastern Africa, Not Specified",
    "501": "Australia",
    "508": "Fiji",
    "511": "Marshall Islands",
    "512": "Micronesia",
    "515": "New Zealand",
    "523": "Tonga",
    "527": "Samoa",
    "554": "Other US Island Areas, Oceania, Not Specified, or at Sea"
}
df['place-of-birth'] = df['place-of-birth'].map(lambda x: pobp_dict.get(str(int(x))) if pd.notna(x) else "N/A")

relp_dict = {
    "0": "Reference person", 
    "1": "Husband/wife", 
    "2": "Biological son or daughter", 
    "3": "Adopted son or daughter", 
    "4": "Stepson or stepdaughter", 
    "5": "Brother or sister", 
    "6": "Father or mother", 
    "7": "Grandchild", 
    "8": "Parent-in-law", 
    "9": "Son-in-law or daughter-in-law", 
    "10": "Other relative", 
    "11": "Roomer or boarder", 
    "12": "Housemate or roommate", 
    "13": "Unmarried partner", 
    "14": "Foster child", 
    "15": "Other nonrelative", 
    "16": "Institutionalized group quarters population", 
    "17": "Noninstitutionalized group quarters population" 
}
df['relationship'] = df['relationship'].map(lambda x: relp_dict.get(str(int(x))) if pd.notna(x) else "N/A")

sex_dict = {
    "1":"Male",
    "2": "Female"
    }
df['sex'] = df['sex'].map(lambda x: sex_dict.get(str(int(x))) if pd.notna(x) else "N/A")

race_dict = {
    "1": "White",
    "2": "Black",
    "3": "American Indian",
    "4": "Alaska Native",
    "5": "American Indian and Alaska Native",
    "6": "Asian",
    "7": "Native Hawaiian and Other Pacific Islander",
    "8": "Other Race",
    "9": "Two or More Race"
}
df['race'] = df['race'].map(lambda x: race_dict.get(str(int(x))) if pd.notna(x) else "N/A")

print(df.head())


    age                 workclass  \
0  18.0        Private For-Profit   
1  53.0        Federal Government   
2  41.0        Private For-Profit   
3  18.0  Self-Employed (Not Inc.)   
4  21.0        Federal Government   

                                        education marital-status  \
0              Some college, but less than 1 year  Never married   
1                   GED or alternative credential  Never married   
2                     Regular high school diploma  Never married   
3              Some college, but less than 1 year  Never married   
4  One or more years of college credit, no degree  Never married   

                                      occupation place-of-birth  \
0                                       Cashiers        Georgia   
1                Orderlies and Psychiatric Aides        Indiana   
2  Industrial And Refractory Machinery Mechanics        Alabama   
3                             Coaches and Scouts        Alabama   
4                                

### ✂️ Step 5: Dataset Splitting
To evaluate the scalability and reliability of our system, we must measure its performance across different data volumes.

In this step, we shuffle and split our dataset into **logarithmic increments** of `10K`, `100K`, and `1M` rows. Rather than treating one as a sandbox, **all three datasets are treated equally** and subjected to identical Statistical Treatments. This multi-scale approach allows us to:
- **Measure Audit Latency:** Observing how execution time grows relative to dataset size.
- **Evaluate Memory Footprint:** Verifying the stability of our algorithm without system crashes.
- **Compute Detection Reliability:** Calculating Precision, Recall, and F1-Score matrices across varying scales to prove our system produces dependable results regardless of dataset size.

<div align="center">
    <img src="img/6.png">
</div>



In [7]:
sample_sizes = {
    "10K": 10000,
    "100K": 100000,
    "1M": 1000000
}
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

for label, size in sample_sizes.items():
    filename = f"ACSIncome_2018_{label}.csv"
    if os.path.exists(filename):
        print(f"Found {filename}")
    else:
        df_subset = df_shuffled.head(size)
        output_filename = f"ACSIncome_2018_{label}.csv"
        df_subset.to_csv(output_filename, index=False)
            
        print(f"Successfully saved {len(df_subset):,} rows to: {output_filename}")


Successfully saved 10,000 rows to: ACSIncome_2018_10K.csv
Successfully saved 100,000 rows to: ACSIncome_2018_100K.csv
Successfully saved 1,000,000 rows to: ACSIncome_2018_1M.csv


<div align="center">
    <img src="img/8.png">
</div>

<div align="center">
    <img src="img/7.png">
</div>